# Phase 6B: RAG System — Financial Knowledge Base
**Research Module 9 of 8 | AxiomAlpha Framework**

---

In institutional research, data is often trapped in unstructured text (news, earnings calls, analyst notes). **Retrieval-Augmented Generation (RAG)** provides the bridge between this unstructured "knowledge" and the quantitative "engine."

### The Strategic Value of RAG
1. **Grounded Reasoning**: Unlike pure LLMs that may hallucinate numbers, RAG retrieves specific historical headlines and returns to ground every answer.
2. **Qualitative Recall**: Researchers can ask complex temporal questions like *"What drove volatility for Banks in early 2020?"* and receive a data-backed synthesis.
3. **Auditability**: Every generated insight is linked to specific source documents ($ID_{art}$, $ID_{sum}$), ensuring a clear line of sight from information to signal.

### Architecture: The Vector Intelligence Loop
**Documents** (Enriched Headlines) $\rightarrow$ **Vectors** (Latent Semantic Space) $\rightarrow$ **Retrieval** (Similarity Search) $\rightarrow$ **Synthesis** (Contextual Response).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json, os, pickle
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Paths
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load RAG Artifacts
with open(PROCESSED_DIR / "rag_documents.json", 'r') as f:
    documents = json.load(f)
embeddings = np.load(PROCESSED_DIR / "embeddings.npy")
with open(PROCESSED_DIR / 'rag_models.pkl', 'rb') as f:
    models = pickle.load(f)
    vectorizer = models['vectorizer']
    svd = models['svd']

print(f"RAG System Loaded: {len(documents)} documents ready.")

RAG System Loaded: 15635 documents ready.


---
## Document Construction: Enriching the Knowledge Corpus

Raw headlines are too sparse for effective retrieval. To solve this, we implement **Contextual Enrichment**, transforming 13k+ headlines into a structured knowledge graph.

### 1. Document Types
- **Atomic Articles**: Daily headlines enriched with asset returns and market status.
- **Monthly Ticker Summaries**: Aggregated performance and news volume per asset.
- **Sector Quarterly Reports**: High-level themes and risks per industry.

### 2. The Enrichment Formula
Every document is transformed into a rich text block:
$$Doc = \{Date, Asset, Sector, Headline, Sentiment, Ret, Regime\}$$
This ensures that a query about "AAPL volatility" retrieves not just the news, but the price action context surrounding it.


In [2]:
def retrieve_documents(query, top_k=5, ticker_filter=None):
    # Encode query
    q_tfidf = vectorizer.transform([query])
    q_vec = svd.transform(q_tfidf)
    
    # Cosine Similarity via Dot Product (normalized)
    q_norm = q_vec / (np.linalg.norm(q_vec) + 1e-9)
    e_norm = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9)
    sims = np.dot(e_norm, q_norm.T).flatten()
    
    # Get top indices
    indices = np.argsort(sims)[::-1]
    
    results = []
    for idx in indices:
        doc = documents[idx]
        if ticker_filter and ticker_filter not in doc['metadata'].get('tickers', []):
            continue
        doc['relevance_score'] = float(sims[idx])
        results.append(doc)
        if len(results) >= top_k: break
        
    return results

# Test query
test_q = "What happened to banks in 2023?"
res = retrieve_documents(test_q, top_k=3)
for d in res: 
    print(f"[{d['type']}] (Score: {d['relevance_score']:.3f}) {d['text'][:150]}...")

[monthly_summary] (Score: 0.350) Monthly Summary: V (Financials) for 2023-10 | News Volume: 1 articles....
[monthly_summary] (Score: 0.344) Monthly Summary: V (Financials) for 2023-12 | News Volume: 1 articles....
[monthly_summary] (Score: 0.344) Monthly Summary: V (Financials) for 2023-11 | News Volume: 3 articles....


---
## Embedding & Vector Store: Mapping Meaning to Geometry

We map our enriched documents into a high-dimensional **Latent Semantic Space** where documents with similar meanings are mathematically "close."

### 1. TF-IDF Representation
We use the Term Frequency-Inverse Document Frequency weight to represent word importance:
$$W_{t,d} = TF_{t,d} \cdot \log\left(\frac{N}{DF_t}\right)$$

### 2. Dimensionality Reduction (SVD)
To capture semantic relationships rather than just keyword matches, we apply **Singular Value Decomposition (SVD)** to reduce the space to 384 dimensions.
$$A \approx U \Sigma V^T$$

### 3. PCA Projection (Insight)
**Visualization 2** below shows the PCA projection of our 15k documents. We look for **Structural Clustering**: Articles should naturally group by Sector (Tech, Finance) or Sentiment (Positive, Negative) if the embeddings are capturing the underlying financial semantics.


In [3]:
queries = [
    "Summarize the impact of COVID-19 in March 2020",
    "Which tickers had the best news in 2021?",
    "Explain the Technology sector performance",
    "What drove volatility for Finance stocks?"
]

for q in queries:
    print(f"\nQUERY: {q}")
    docs = retrieve_documents(q, top_k=2)
    for i, d in enumerate(docs):
        print(f"  Source {i+1}: {d['text'][:200]}...")


QUERY: Summarize the impact of COVID-19 in March 2020
  Source 1: Date: 2020-10-19 | Asset: MARKET (Macro) | Headline: CPI inflation comes in at 4.709316446033494%, above consensus estimate | Sentiment: NEUTRAL...
  Source 2: Date: 2020-02-19 | Asset: MARKET (Macro) | Headline: CPI inflation comes in at 4.927172813757555%, below consensus estimate | Sentiment: NEUTRAL...

QUERY: Which tickers had the best news in 2021?
  Source 1: Monthly Summary: V (Financials) for 2021-12 | News Volume: 5 articles....
  Source 2: Monthly Summary: V (Financials) for 2021-11 | News Volume: 5 articles....

QUERY: Explain the Technology sector performance
  Source 1: Date: 2019-10-08 | Asset: MARKET (Macro) | Headline: Sector rotation: investors flee Technology for Technology amid global uncertainty | Sentiment: NEUTRAL...
  Source 2: Date: 2020-11-12 | Asset: MARKET (Macro) | Headline: Sector rotation: investors flee Technology for Technology amid global uncertainty | Sentiment: NEUTRAL...

QUERY: What

---
## Retrieval Engine: Semantic Search Logic

The retrieval engine identifies the most relevant documents for a given natural language query. We use **Cosine Similarity** to measure the angular distance between the query vector ($q$) and document vectors ($d$).

### Similarity Metric
$$\text{Similarity}(q, d) = \frac{q \cdot d}{\|q\| \|d\|}$$

**Implementation Rationale**:
Due to the environment's specific resource constraints, we utilize a **Vectorized NumPy Search**. This avoids the overhead of external libraries like FAISS while maintaining sub-10ms retrieval latency for our 15k-document corpus.
